# Diabetes Risk Prediction — Full Analysis Notebook

**Course:** Introduction to Data Science · Semester 2  
**Group:** Ali Raza (2540010) · Muhammad Ammar (2540008) · Taha Ali (2540008)  
**Instructor:** Sir Zaki

---

## What this notebook does
We train a **Random Forest classifier** (supervised binary classification) on the **BRFSS 2015 Diabetes Health Indicators** dataset (~70 000 patients, 21 features) to predict whether a person is at risk of diabetes.

### Steps
1. Load the data
2. Inspect (size, describe, missing, dtypes)
3. Clean (drop duplicates, fix dtypes)
4. EDA — visualisations
5. Correlation analysis
6. Train / test split (80 / 20, stratified)
7. Train Random Forest
8. Evaluate (accuracy + confusion matrix + classification report)
9. Feature importance → top 5 risk factors
10. Save model for the Streamlit app


## 0. Setup

In [ ]:
# If something is missing, uncomment and run:
# !pip install pandas numpy matplotlib seaborn scikit-learn joblib

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
TEAL, AMBER = '#0F766E', '#F59E0B'

# Paths — assume notebook lives in  notebooks/  inside the project folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH    = PROJECT_ROOT / 'data' / 'diabetes_binary_5050split_health_indicators_BRFSS2015.csv'
MODELS_DIR   = PROJECT_ROOT / 'models'; MODELS_DIR.mkdir(exist_ok=True)
PLOTS_DIR    = PROJECT_ROOT / 'plots';  PLOTS_DIR.mkdir(exist_ok=True)
print('Project root:', PROJECT_ROOT)

## 1. Load the data

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

## 2. Inspect

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nClass balance:')
print(df['Diabetes_binary'].value_counts(normalize=True).round(3))

## 3. Clean — drop duplicates, fix dtypes

In [ ]:
print('Duplicates before:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('Duplicates after :', df.duplicated().sum())
print('Shape after cleaning:', df.shape)

# Convert binary / ordinal columns to int (everything except BMI)
int_cols = [c for c in df.columns if c != 'BMI']
df[int_cols] = df[int_cols].astype(int)
df.dtypes

## 4. Exploratory Data Analysis

In [ ]:
# 4.1  Class balance
plt.figure(figsize=(5, 3.2))
sns.countplot(x='Diabetes_binary', data=df, palette=[TEAL, AMBER])
plt.title('Class balance  (0 = No Diabetes, 1 = Diabetes)')
plt.tight_layout(); plt.savefig(PLOTS_DIR / '01_class_balance.png'); plt.show()

In [ ]:
# 4.2  BMI distribution by class
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x='BMI', hue='Diabetes_binary',
             bins=40, kde=True, palette=[TEAL, AMBER])
plt.title('BMI distribution by Diabetes status')
plt.tight_layout(); plt.savefig(PLOTS_DIR / '02_bmi_distribution.png'); plt.show()

In [ ]:
# 4.3  BMI boxplot
plt.figure(figsize=(5, 4))
sns.boxplot(x='Diabetes_binary', y='BMI', data=df, palette=[TEAL, AMBER])
plt.title('BMI by Diabetes status')
plt.tight_layout(); plt.savefig(PLOTS_DIR / '03_bmi_boxplot.png'); plt.show()

In [ ]:
# 4.4  Bar charts of HighBP and HighChol
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.countplot(x='HighBP', hue='Diabetes_binary', data=df, palette=[TEAL, AMBER], ax=axes[0])
axes[0].set_title('High Blood Pressure')
sns.countplot(x='HighChol', hue='Diabetes_binary', data=df, palette=[TEAL, AMBER], ax=axes[1])
axes[1].set_title('High Cholesterol')
plt.tight_layout(); plt.savefig(PLOTS_DIR / '04_bp_chol.png'); plt.show()

## 5. Correlation analysis

In [ ]:
corr = df.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, cbar_kws={'shrink': 0.75})
plt.title('Correlation matrix of all features')
plt.tight_layout(); plt.savefig(PLOTS_DIR / '05_correlation_heatmap.png'); plt.show()

In [ ]:
# Top 10 features most correlated with Diabetes
top_corr = (corr['Diabetes_binary'].drop('Diabetes_binary')
            .abs().sort_values(ascending=False).head(10))
print('Top 10 features most correlated with Diabetes:')
print(top_corr.round(3))

## 6. Train / Test split (do this BEFORE preprocessing)

> 🔒 Splitting first prevents **data leakage** — a serious viva-killer mistake.

In [ ]:
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
print('Train:', X_train.shape, '  Test:', X_test.shape)

## 7. Train Random Forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)
print('Training done.')

## 8. Evaluate

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'TEST ACCURACY : {acc:.4f}   ({acc*100:.2f}%)')

cm = confusion_matrix(y_test, y_pred)
print('\nConfusion matrix:')
print(pd.DataFrame(cm,
                   index=['actual: No Diabetes', 'actual: Diabetes'],
                   columns=['pred: No Diabetes', 'pred: Diabetes']))

print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['No Diabetes','Diabetes']))

In [ ]:
# Pretty confusion matrix plot
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(cm,
    display_labels=['No Diabetes', 'Diabetes']).plot(ax=ax, cmap='BuGn', colorbar=False)
plt.title('Confusion Matrix')
plt.tight_layout(); plt.savefig(PLOTS_DIR / '06_confusion_matrix.png'); plt.show()

## 9. Feature importance — top 5 risk factors

In [ ]:
importance = (pd.Series(model.feature_importances_, index=X.columns)
                .sort_values(ascending=True))

plt.figure(figsize=(8, 8))
importance.plot(kind='barh', color=TEAL)
plt.title('Random Forest — Feature importance')
plt.xlabel('Importance score')
plt.tight_layout(); plt.savefig(PLOTS_DIR / '07_feature_importance.png'); plt.show()

print('Top 5 diabetes risk factors:')
print(importance.sort_values(ascending=False).head(5).round(4))

## 10. Save the model + feature names

Saving the feature order is critical — the Streamlit app must feed inputs in the same order, otherwise predictions will be wrong.

In [ ]:
joblib.dump(model, MODELS_DIR / 'rf_model.joblib')
with open(MODELS_DIR / 'feature_names.json', 'w') as f:
    json.dump(list(X.columns), f, indent=2)
with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump({
        'test_accuracy': round(float(acc), 4),
        'test_size':     int(len(y_test)),
        'n_features':    int(X.shape[1]),
        'top_5_features': importance.sort_values(ascending=False).head(5).round(4).to_dict(),
    }, f, indent=2)
print('Saved:')
print('  models/rf_model.joblib')
print('  models/feature_names.json')
print('  models/metrics.json')

## ✅ Done

Now you can launch the Streamlit GUI:

```bash
cd ..       # go back to project root
streamlit run app/streamlit_app.py
```

Your browser will open at http://localhost:8501 with the prediction form.